In [1]:
from scipy.stats import mannwhitneyu
import pandas as pd
import csv

In [3]:
def read_mean_mAP50(individual, model_history_paths):
    results = {}
    dataFrame = pd.read_csv(model_history_paths)
    if individual: 
        dataframe = dataFrame[dataFrame['threshold'] != 0.001]
    else: # exclude the thresholds that were not statistically significant different to the pivot
        dataframe = dataFrame[~dataFrame['threshold'].isin([0.001, 0.1, 0.2, 0.3])]
    max_mAP50_index = dataframe['metrics/mAP50(B)'].idxmax()
    max_mAP50 = dataframe.loc[max_mAP50_index, 'metrics/mAP50(B)']
    threshold = dataframe.loc[max_mAP50_index, 'threshold']
    results = (max_mAP50, threshold)
    return results

def mann_whitney(dataset, pivot_threshold, pivot_vector, pivot_model, remining_models, results_path):
    with open(results_path, 'w', newline='') as csvfile:
        columns = ["Pivot model name", "Pivot model threshold", "Pivot model mean", "Other model name", "Other model threshold", "Other model mean", "p-value"]
        writer = csv.writer(csvfile)
        writer.writerow(columns)

        pivot_mean_mAP50 = pivot_vector.mean()

        for model in remining_models:
            for threshold in remining_models[model]:
                df = pd.DataFrame(remining_models[model][threshold])
                mean_mAP50 = df.mean().iloc[0]
                result = mannwhitneyu(pivot_vector, remining_models[model][threshold])
                p_value = result.pvalue
                if p_value < 0.05:
                    answer = "Significant"
                else :
                    answer = "Not significant"
                print(f"P-value for {pivot_model} {dataset} vs {model}, {threshold}: {p_value} - {answer}")
                values = [str(pivot_model), str(pivot_threshold), str(pivot_mean_mAP50), str(model), str(threshold), str(mean_mAP50), str(p_value)]
                writer.writerow(values)

def run_dataset(includeModel, thresholds, dataset, dataset_mean_path, pivot_model, models, results_path, user_path):
    model_maxmAP50_threshold = read_mean_mAP50(includeModel, dataset_mean_path)
    vectors = {}
    pivot_vector = {}

    path = user_path + '/Breast_Cancer_Investigation/DetectionModels/runs_test/' + dataset + '/' + pivot_model + '.pt_' + str(model_maxmAP50_threshold[1]) + '.csv'
    df = pd.read_csv(path)
    pivot_vector = df['metrics/mAP50(B)'] 
 
    if includeModel:
        vectors[models] = {}
        for threshold in thresholds:
            path = user_path + '/Breast_Cancer_Investigation/DetectionModels/runs_test/' + dataset + '/' + models + '.pt_' + str(threshold) + '.csv'
            df = pd.read_csv(path)
            vectors[models][threshold] = df['metrics/mAP50(B)'].tolist() 
    else:   
        for model in models:
            if model != pivot_model:
                vectors[model] = {}
                for threshold in thresholds:
                    path = user_path + '/Breast_Cancer_Investigation/DetectionModels/runs_test/' + dataset + '/' + model + '.pt_' + str(threshold) + '.csv'
                    df = pd.read_csv(path)
                    vectors[model][threshold] = df['metrics/mAP50(B)'].tolist()           

    mann_whitney(dataset, model_maxmAP50_threshold[1], pivot_vector, pivot_model, vectors, results_path)

# INbreast

In [4]:
user_path = 'C:/Users/camiz' # Change this path to your path

INbreast_models_mean_path = {
    'yolo11n': user_path + '/Breast_Cancer_Investigation/DetectionModels/runs_test/INbreast/yolo11n.pt/metrics.csv',
    'yolo11s': user_path + '/Breast_Cancer_Investigation/DetectionModels/runs_test/INbreast/yolo11s.pt/metrics.csv',
    'yolo11m': user_path + '/Breast_Cancer_Investigation/DetectionModels/runs_test/INbreast/yolo11m.pt/metrics.csv'
}

models = ['yolo11n', 'yolo11s', 'yolo11m']

Individual testing to see if between the threshold of the pivot model there is a significant statistic difference

In [5]:
thresholds = {0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9}

print("------- Statistical test for the YOLOv11 Nano model ----------")
model = 'yolo11n'
INbreast_result_path = user_path + '/Breast_Cancer_Investigation/DetectionModels/ComparisonStadistic/INbreast/' + model + '_stadistic_individual_resultsMW.csv'
run_dataset(True, thresholds, 'INbreast', INbreast_models_mean_path[model], model, model, INbreast_result_path, user_path)

------- Statistical test for the YOLOv11 Nano model ----------
P-value for yolo11n INbreast vs yolo11n, 0.1: 1.0 - Not significant
P-value for yolo11n INbreast vs yolo11n, 0.4: 0.13956906564539745 - Not significant
P-value for yolo11n INbreast vs yolo11n, 0.2: 0.6766659743526438 - Not significant
P-value for yolo11n INbreast vs yolo11n, 0.3: 0.2555846160278239 - Not significant
P-value for yolo11n INbreast vs yolo11n, 0.5: 0.033821522780542006 - Significant
P-value for yolo11n INbreast vs yolo11n, 0.6: 0.00908242666764077 - Significant
P-value for yolo11n INbreast vs yolo11n, 0.7: 0.0007580239933292277 - Significant
P-value for yolo11n INbreast vs yolo11n, 0.8: 0.00017861448837368162 - Significant
P-value for yolo11n INbreast vs yolo11n, 0.9: 0.00017761066068896375 - Significant


Testing for all the models, but only with the thresholds that are going to be used

In [6]:
thresholds = {0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9}

print("------- Statistical test for the YOLOv11 Nano model ----------")
model = 'yolo11n'
INbreast_result_path = user_path + '/Breast_Cancer_Investigation/DetectionModels/ComparisonStadistic/INbreast/' + model + '_stadistic_resultsMW.csv'
run_dataset(False, thresholds, 'INbreast', INbreast_models_mean_path[model], model, models, INbreast_result_path, user_path)

print("------- Statistical test for the YOLOv11 Small model ----------")
model = 'yolo11s'
INbreast_result_path = user_path + '/Breast_Cancer_Investigation/DetectionModels/ComparisonStadistic/INbreast/' + model + '_stadistic_resultsMW.csv'
run_dataset(False, thresholds,'INbreast', INbreast_models_mean_path[model], model, models, INbreast_result_path, user_path)

print("------- Statistical test for the YOLOv11 Medium model ----------")
model = 'yolo11m'
INbreast_result_path = user_path + '/Breast_Cancer_Investigation/DetectionModels/ComparisonStadistic/INbreast/' + model + '_stadistic_resultsMW.csv'
run_dataset(False, thresholds,'INbreast', INbreast_models_mean_path[model], model, models, INbreast_result_path, user_path)

------- Statistical test for the YOLOv11 Nano model ----------
P-value for yolo11n INbreast vs yolo11s, 0.3: 0.8796944322344019 - Not significant
P-value for yolo11n INbreast vs yolo11s, 0.4: 0.7911833481406884 - Not significant
P-value for yolo11n INbreast vs yolo11s, 0.6: 0.034226150579334216 - Significant
P-value for yolo11n INbreast vs yolo11s, 0.5: 0.36380450713550716 - Not significant
P-value for yolo11n INbreast vs yolo11s, 0.7: 0.0010035563801035307 - Significant
P-value for yolo11n INbreast vs yolo11s, 0.8: 0.0003264344499015525 - Significant
P-value for yolo11n INbreast vs yolo11s, 0.9: 0.00013172960306213278 - Significant
P-value for yolo11n INbreast vs yolo11m, 0.3: 0.4961297149428171 - Not significant
P-value for yolo11n INbreast vs yolo11m, 0.4: 0.570605503511469 - Not significant
P-value for yolo11n INbreast vs yolo11m, 0.6: 0.3067609561706547 - Not significant
P-value for yolo11n INbreast vs yolo11m, 0.5: 0.18587673236587576 - Not significant
P-value for yolo11n INbreas

# CBIS

In [1]:
user_path = 'C:/Users/camiz' # Change this path to your path

CBIS_models_mean_path = {
    'yolo11n': user_path + '/Breast_Cancer_Investigation/DetectionModels/runs_test/CBIS-DDSM/yolo11n.pt/metrics.csv',
    'yolo11s': user_path + '/Breast_Cancer_Investigation/DetectionModels/runs_test/CBIS-DDSM/yolo11s.pt/metrics.csv',
    'yolo11m': user_path + '/Breast_Cancer_Investigation/DetectionModels/runs_test/CBIS-DDSM/yolo11m.pt/metrics.csv'
}

models = ['yolo11n', 'yolo11s', 'yolo11m']

Individual testing to see if between the threshold of the pivot model there is a significant statistic difference

In [5]:
thresholds = {0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9}

print("------- Statistical test for the YOLOv11 Nano model ----------")
model = 'yolo11n'
CBIS_result_path = user_path + '/Breast_Cancer_Investigation/DetectionModels/ComparisonStadistic/CBIS-DDSM/' + model + '_stadistic_individual_resultsMW.csv'
run_dataset(True, thresholds, 'CBIS-DDSM', CBIS_models_mean_path[model], model, model, CBIS_result_path, user_path)

------- Statistical test for the YOLOv11 Nano model ----------
P-value for yolo11n CBIS-DDSM vs yolo11n, 0.1: 1.0 - Not significant
P-value for yolo11n CBIS-DDSM vs yolo11n, 0.4: 0.04515456962427901 - Significant
P-value for yolo11n CBIS-DDSM vs yolo11n, 0.2: 0.7337299956962472 - Not significant
P-value for yolo11n CBIS-DDSM vs yolo11n, 0.3: 0.3846730627355087 - Not significant
P-value for yolo11n CBIS-DDSM vs yolo11n, 0.5: 0.014019277113959953 - Significant
P-value for yolo11n CBIS-DDSM vs yolo11n, 0.6: 0.0010079762403767444 - Significant
P-value for yolo11n CBIS-DDSM vs yolo11n, 0.7: 0.00018267179110955002 - Significant
P-value for yolo11n CBIS-DDSM vs yolo11n, 0.8: 0.000181651146091465 - Significant
P-value for yolo11n CBIS-DDSM vs yolo11n, 0.9: 0.00011067076459386204 - Significant


Testing for all the models, but only with the thresholds that are going to be used

In [6]:
thresholds = {0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9}

print("------- Statistical test for the YOLOv11 Nano model ----------")
model = 'yolo11n'
CBIS_result_path = user_path + '/Breast_Cancer_Investigation/DetectionModels/ComparisonStadistic/CBIS-DDSM/' + model + '_stadistic_resultsMW.csv'
run_dataset(False, thresholds, 'CBIS-DDSM', CBIS_models_mean_path[model], model, models, CBIS_result_path, user_path)

print("------- Statistical test for the YOLOv11 Small model ----------")
model = 'yolo11s'
CBIS_result_path = user_path + '/Breast_Cancer_Investigation/DetectionModels/ComparisonStadistic/CBIS-DDSM/' + model + '_stadistic_resultsMW.csv'
run_dataset(False, thresholds,'CBIS-DDSM', CBIS_models_mean_path[model], model, models, CBIS_result_path, user_path)

print("------- Statistical test for the YOLOv11 Medium model ----------")
model = 'yolo11m'
CBIS_result_path = user_path + '/Breast_Cancer_Investigation/DetectionModels/ComparisonStadistic/CBIS-DDSM/' + model + '_stadistic_resultsMW.csv'
run_dataset(False, thresholds,'CBIS-DDSM', CBIS_models_mean_path[model], model, models, CBIS_result_path, user_path)

------- Statistical test for the YOLOv11 Nano model ----------
P-value for yolo11n CBIS-DDSM vs yolo11s, 0.3: 0.6231762238821174 - Not significant
P-value for yolo11n CBIS-DDSM vs yolo11s, 0.4: 0.24132159301718004 - Not significant
P-value for yolo11n CBIS-DDSM vs yolo11s, 0.6: 0.004586392080253494 - Significant
P-value for yolo11n CBIS-DDSM vs yolo11s, 0.5: 0.04515456962427901 - Significant
P-value for yolo11n CBIS-DDSM vs yolo11s, 0.7: 0.00024480482452445495 - Significant
P-value for yolo11n CBIS-DDSM vs yolo11s, 0.8: 0.00014939276635154945 - Significant
P-value for yolo11n CBIS-DDSM vs yolo11s, 0.9: 8.744986536701464e-05 - Significant
P-value for yolo11n CBIS-DDSM vs yolo11m, 0.3: 0.5205228832757727 - Not significant
P-value for yolo11n CBIS-DDSM vs yolo11m, 0.4: 0.9698499769931556 - Not significant
P-value for yolo11n CBIS-DDSM vs yolo11m, 0.6: 0.0013149446697132139 - Significant
P-value for yolo11n CBIS-DDSM vs yolo11m, 0.5: 0.14046504815835495 - Not significant
P-value for yolo11